# 🧳 Умный помощник по командировкам — мультиагентная система (LangGraph)

**ДЗ №3. RAG + Multi-Agent System.**

> ⚠️ **Инструмент для менеджера отдела деловых поездок**, а не для сотрудника-заказчика.
> Сотрудник лишь подаёт заявку; менеджер деловых поездок получает её, **запускает процесс агента**,
> проверяет предложенные варианты и принимает финальное решение.

Подсистема автоматизирует оформление командировки:

1. 🧭 **Менеджер** строит план и делегирует задачи (*Plan-and-Execute*).
2. 📜 **Агент политики** через **RAG** достаёт правила из корпоративной базы знаний.
3. 🔎 **Поисковик** ищет билеты и отели и **предлагает** варианты (*ReAct + web search*).
4. 💰 **Аналитик бюджета** сверяет варианты с лимитами политики.
5. 🙋 **Менеджер отдела деловых поездок** выбирает финальный вариант (*Human-in-the-loop*).
6. 📧 **Нотификатор** «отправляет» итог на почту — **заглушка** (ничего реально не отправляется).

Сквозь все шаги работает 🔭 **Observability** — трассировка действий агентов.

> Нотбук запускается **без API-ключей** (мок-LLM + TF-IDF RAG + мок-поиск).
> Если задать `OPENAI_API_KEY` / `TAVILY_API_KEY` — включатся настоящие LLM, эмбеддинги и web-поиск.


## 1. Установка зависимостей

In [ ]:
%%capture
# В Google Colab выполнить один раз. scikit-learn и numpy обычно уже стоят.
!pip install -q langgraph langchain langchain-core langchain-openai langchain-community scikit-learn numpy


## 2. Конфигурация и ключи

Оставьте поля пустыми — нотбук пойдёт по «оффлайн»-пути (мок-LLM, локальный RAG, мок-поиск).
Заполните, чтобы включить реальные сервисы.

In [ ]:
import os

# Вставьте ключи, если хотите реальные LLM/эмбеддинги/web-поиск (необязательно):
os.environ.setdefault("OPENAI_API_KEY", "")   # для ChatOpenAI + эмбеддингов
os.environ.setdefault("TAVILY_API_KEY", "")    # для реального web search

USE_OPENAI = bool(os.environ.get("OPENAI_API_KEY"))
USE_TAVILY = bool(os.environ.get("TAVILY_API_KEY"))

print(f"USE_OPENAI={USE_OPENAI}  USE_TAVILY={USE_TAVILY}")
print("Оффлайн-режим" if not USE_OPENAI else "Онлайн-режим (OpenAI)")


## 3. RAG-контекст: мини-база знаний (политика командировок)

Это **придуманная** корпоративная политика. Источник данных для RAG.
В проде лежала бы в Confluence / PDF / Markdown и индексировалась бы в Vector DB.

In [ ]:
# Каждый элемент = смысловой раздел политики (так удобно чанковать: 1 чанк = 1 правило)
TRAVEL_POLICY = [
    {"section": "Общие положения",
     "text": "Командировка оформляется не позднее чем за 7 календарных дней до поездки. "
             "Все расходы возмещаются по чекам в течение 5 рабочих дней после возвращения."},
    {"section": "Перелёты",
     "text": "Для перелётов короче 6 часов разрешён только эконом-класс. Бизнес-класс допускается "
             "при длительности перелёта свыше 6 часов или для сотрудников уровня C-level. "
             "Лимит стоимости перелёта по Европе — не более 600 EUR за поездку (туда-обратно)."},
    {"section": "Проживание",
     "text": "Лимит на отель по Европе — не более 250 EUR за ночь, по РФ и СНГ — не более 180 EUR за ночь. "
             "Категория отеля — не выше 4 звёзд. Желательно наличие завтрака в стоимости."},
    {"section": "Суточные",
     "text": "Суточные (per diem) по Европе составляют 60 EUR в день и покрывают питание и мелкие расходы."},
    {"section": "Бронирование и партнёры",
     "text": "Предпочтительны партнёрские агрегаторы компании. Возврат/обмен билетов должен быть возможен. "
             "Бронь отеля — с бесплатной отменой не позднее чем за сутки."},
]

# Структурированные лимиты (зеркало политики) — их использует Аналитик бюджета.
# В реальной системе их можно извлекать из политики или хранить в Knowledge Graph.
POLICY_LIMITS = {
    "flight_max_eur": 600,
    "hotel_max_eur_per_night_eu": 250,
    "hotel_max_stars": 4,
    "allowed_flight_class": {"economy"},
}

print(f"Разделов в базе знаний: {len(TRAVEL_POLICY)}")


## 4. RAG-пайплайн (внутри Агента политики)

`Чанкинг → Эмбеддинг → Vector store → Retrieve(top-k) → Rerank(top-n)`

- **Эмбеддинг:** OpenAI `text-embedding-3-small`, если есть ключ; иначе TF-IDF (всегда работает).
- **Vector store:** простой numpy-индекс + cosine (роль FAISS/Chroma/pgvector).
- **Reranker:** лёгкий cross-encoder-заменитель на основе пересечения токенов (демонстрация идеи).

In [ ]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

# ----- 4.1 Чанкинг: здесь 1 раздел = 1 чанк (для длинных документов резали бы по ~500 токенов с overlap)
def chunk_policy(policy):
    return [{"id": i, "section": p["section"], "text": f'{p["section"]}: {p["text"]}'}
            for i, p in enumerate(policy)]

CHUNKS = chunk_policy(TRAVEL_POLICY)


# ----- 4.2 Эмбеддинг (реальный или TF-IDF-фолбэк)
class EmbeddingModel:
    def __init__(self, texts):
        self.mode = "openai" if USE_OPENAI else "tfidf"
        if self.mode == "openai":
            from langchain_openai import OpenAIEmbeddings
            self._emb = OpenAIEmbeddings(model="text-embedding-3-small")
        else:
            self._tfidf = TfidfVectorizer().fit(texts)

    def encode(self, texts):
        if self.mode == "openai":
            return np.array(self._emb.embed_documents(list(texts)))
        return self._tfidf.transform(texts).toarray()

    def encode_one(self, text):
        return self.encode([text])[0]


# ----- 4.3 Vector store (роль Vector DB)
class VectorStore:
    def __init__(self, chunks):
        self.chunks = chunks
        self.model = EmbeddingModel([c["text"] for c in chunks])
        self.matrix = self.model.encode([c["text"] for c in chunks])

    @staticmethod
    def _cosine(a, b):
        denom = (np.linalg.norm(a) * np.linalg.norm(b, axis=1) + 1e-9)
        return (b @ a) / denom

    # Retrieve: top-k по cosine
    def retrieve(self, query, k=4):
        q = self.model.encode_one(query)
        scores = self._cosine(q, self.matrix)
        idx = np.argsort(-scores)[:k]
        return [(self.chunks[i], float(scores[i])) for i in idx]


# ----- 4.4 Reranker (упрощённый cross-encoder: пересечение токенов запрос/чанк)
def rerank(query, candidates, n=2):
    q_tokens = set(re.findall(r"\w+", query.lower()))
    def score(item):
        chunk, base = item
        c_tokens = set(re.findall(r"\w+", chunk["text"].lower()))
        overlap = len(q_tokens & c_tokens) / (len(q_tokens) + 1e-9)
        return 0.5 * base + 0.5 * overlap          # смесь dense-score и лексического сигнала
    return sorted(candidates, key=score, reverse=True)[:n]


VECTOR_DB = VectorStore(CHUNKS)


def rag_search(query, k=4, n=2):
    """RAG: retrieve top-k -> rerank top-n. Возвращает текст контекста."""
    cands = VECTOR_DB.retrieve(query, k=k)
    top = rerank(query, cands, n=n)
    return "\n".join(f"- [{c['section']}] {c['text']}" for c, _ in top)


# дымовой тест RAG
print(rag_search("какой лимит на отель в Европе и класс перелёта?"))


## 5. LLM-помощник (реальный или мок)

Логика агентов детерминирована и работает без LLM. LLM используется только для «человеческих»
формулировок (план, финальное письмо). Без ключа — аккуратный шаблонный мок.

In [ ]:
def llm(prompt: str) -> str:
    if USE_OPENAI:
        from langchain_openai import ChatOpenAI
        model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        return model.invoke(prompt).content
    # Мок: возвращает усечённый эхо-ответ, чтобы нотбук работал офлайн
    return "[MOCK-LLM] " + prompt.strip().split("\n")[0][:160]

print(llm("Сформулируй короткий план командировки Москва-Берлин"))


## 6. Состояние графа и Observability

`AgentState` — общая «доска», через которую агенты обмениваются сообщениями.
`trace` — список событий обсервабилити.

In [ ]:
from typing import TypedDict, List, Dict, Any, Optional
import datetime


class AgentState(TypedDict, total=False):
    request: str                       # заявка сотрудника
    plan: List[str]                    # план Менеджера
    policy_context: str                # результат RAG (Агент политики)
    options: List[Dict[str, Any]]      # варианты Поисковика
    compliant: List[Dict[str, Any]]    # прошедшие проверку бюджета
    human_choice: Optional[str]        # выбор человека (HITL)
    email_status: str                  # статус отправки (mock)
    trace: List[Dict[str, str]]        # обсервабилити


def log(state: AgentState, agent: str, message: str):
    """Записать событие обсервабилити в общий trace."""
    ts = datetime.datetime.now().strftime("%H:%M:%S")
    state.setdefault("trace", []).append({"ts": ts, "agent": agent, "msg": message})
    print(f"[{ts}] {agent:<18} | {message}")


## 7. Агенты (узлы графа)

Каждый узел — отдельный агент с единственной зоной ответственности (Single Responsibility).

In [ ]:
# 7.1 🧭 Менеджер — планирование и делегирование (Plan-and-Execute)
def planner_node(state: AgentState) -> AgentState:
    plan = [
        "Запросить корпоративную политику (RAG)",
        "Найти билеты и отели (web search)",
        "Проверить варианты на соответствие бюджету",
        "Передать варианты человеку для выбора",
        "Отправить итог на почту (mock)",
    ]
    state["plan"] = plan
    log(state, "Менеджер", f"План построен из {len(plan)} шагов")
    _ = llm(f"Заявка: {state['request']}. Кратко поясни план командировки.")
    return state


In [ ]:
# 7.2 📜 Агент политики — RAG внутри агента
def policy_node(state: AgentState) -> AgentState:
    query = f"лимиты на отель, класс и стоимость перелёта, суточные для: {state['request']}"
    context = rag_search(query, k=4, n=3)
    state["policy_context"] = context
    log(state, "Агент политики", f"RAG вернул {context.count(chr(10)) + 1} релевантных правил")
    return state


In [ ]:
# 7.3 🔎 Поисковик — предлагает варианты (ReAct + web search или мок)
def _mock_search_results():
    return [
        {"id": "A", "flight_class": "economy", "flight_eur": 420,
         "hotel": "Park Inn Berlin", "stars": 3, "hotel_eur_night": 180, "nights": 3},
        {"id": "B", "flight_class": "economy", "flight_eur": 380,
         "hotel": "Motel One Berlin", "stars": 3, "hotel_eur_night": 150, "nights": 3},
        {"id": "C", "flight_class": "business", "flight_eur": 1500,
         "hotel": "Grand Luxe Berlin", "stars": 5, "hotel_eur_night": 600, "nights": 3},
    ]


def searcher_node(state: AgentState) -> AgentState:
    if USE_TAVILY:
        # Реальный поиск: тут можно дернуть TavilySearchResults и распарсить выдачу.
        # Для надёжности демо всё равно используем структурированные варианты.
        from langchain_community.tools.tavily_search import TavilySearchResults
        try:
            _ = TavilySearchResults(max_results=3).invoke(
                f"авиабилеты и отели для командировки: {state['request']}")
            log(state, "Поисковик", "Web search выполнен (Tavily)")
        except Exception as e:
            log(state, "Поисковик", f"Tavily недоступен ({e}); беру мок-варианты")
    options = _mock_search_results()
    for o in options:
        o["total_eur"] = o["flight_eur"] + o["hotel_eur_night"] * o["nights"]
    state["options"] = options
    log(state, "Поисковик", f"Предложено вариантов: {len(options)}")
    return state


In [ ]:
# 7.4 💰 Аналитик бюджета — сверка с лимитами политики
def budget_node(state: AgentState) -> AgentState:
    compliant = []
    for o in state["options"]:
        reasons = []
        if o["flight_class"] not in POLICY_LIMITS["allowed_flight_class"]:
            reasons.append(f"класс {o['flight_class']} запрещён")
        if o["flight_eur"] > POLICY_LIMITS["flight_max_eur"]:
            reasons.append(f"перелёт {o['flight_eur']}€ > {POLICY_LIMITS['flight_max_eur']}€")
        if o["hotel_eur_night"] > POLICY_LIMITS["hotel_max_eur_per_night_eu"]:
            reasons.append(f"отель {o['hotel_eur_night']}€/ночь > лимита")
        if o["stars"] > POLICY_LIMITS["hotel_max_stars"]:
            reasons.append(f"{o['stars']}★ > {POLICY_LIMITS['hotel_max_stars']}★")
        verdict = "OK" if not reasons else "; ".join(reasons)
        log(state, "Аналитик бюджета", f"Вариант {o['id']}: {verdict}")
        if not reasons:
            compliant.append(o)
    state["compliant"] = compliant
    log(state, "Аналитик бюджета", f"Прошли политику: {[o['id'] for o in compliant]}")
    return state


In [ ]:
# 7.5 🙋 Human-in-the-loop — финальный выбор за человеком
# Задайте HUMAN_CHOICE = "A"/"B"/... чтобы сымитировать выбор без ввода с клавиатуры.
HUMAN_CHOICE = None  # None -> авто-выбор первого compliant (имитация одобрения человеком)


def human_node(state: AgentState) -> AgentState:
    compliant = state.get("compliant", [])
    if not compliant:
        state["human_choice"] = None
        log(state, "Human-in-the-loop", "Нет вариантов в рамках политики — нужен пересмотр")
        return state

    ids = [o["id"] for o in compliant]
    choice = HUMAN_CHOICE
    if choice is None:
        # В реальном Colab можно заменить на: choice = input(f"Выберите вариант {ids}: ")
        choice = ids[0]
        log(state, "Human-in-the-loop", f"(имитация) менеджер деловых поездок выбрал вариант {choice} из {ids}")
    else:
        log(state, "Human-in-the-loop", f"менеджер деловых поездок выбрал вариант {choice} из {ids}")
    state["human_choice"] = choice if choice in ids else ids[0]
    return state


In [ ]:
# 7.6 📧 Нотификатор — отправка на почту (ЗАГЛУШКА: ничего не отправляется)
def notifier_node(state: AgentState) -> AgentState:
    choice = state.get("human_choice")
    if not choice:
        state["email_status"] = "skipped"
        log(state, "Нотификатор", "Нечего отправлять")
        return state

    opt = next(o for o in state["compliant"] if o["id"] == choice)
    body = llm(
        f"Сформируй короткое письмо сотруднику о брони командировки. "
        f"Вариант {opt['id']}: перелёт {opt['flight_class']} {opt['flight_eur']}€, "
        f"отель {opt['hotel']} {opt['stars']}★ {opt['hotel_eur_night']}€/ночь x{opt['nights']}, "
        f"итого {opt['total_eur']}€."
    )

    # ---- MOCK SEND: реальная отправка отключена ----
    print("\n===== 📧 MOCK EMAIL (НЕ отправляется реально) =====")
    print("To: employee@company.com")
    print("Subject: Командировка — выбранный вариант")
    print(body)
    print(f"Детали брони: {opt}")
    print("====================================================\n")

    state["email_status"] = f"mock-sent (вариант {choice})"
    log(state, "Нотификатор", state["email_status"])
    return state


## 8. Сборка графа LangGraph

Линейный конвейер с делегированием: `planner → policy → searcher → budget → human → notifier`.
Менеджер (planner) задаёт план, остальные узлы — исполнители, общающиеся через общий `AgentState`.

In [ ]:
from langgraph.graph import StateGraph, START, END

builder = StateGraph(AgentState)
builder.add_node("planner", planner_node)
builder.add_node("policy", policy_node)
builder.add_node("searcher", searcher_node)
builder.add_node("budget", budget_node)
builder.add_node("human", human_node)
builder.add_node("notifier", notifier_node)

builder.add_edge(START, "planner")
builder.add_edge("planner", "policy")
builder.add_edge("policy", "searcher")
builder.add_edge("searcher", "budget")
builder.add_edge("budget", "human")
builder.add_edge("human", "notifier")
builder.add_edge("notifier", END)

graph = builder.compile()
print("Граф собран. Узлы:", list(graph.get_graph().nodes))


### (Опционально) Визуализация графа средствами LangGraph

In [ ]:
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print("Не удалось отрисовать PNG (нужен интернет/graphviz). Mermaid-код ниже:\n")
    print(graph.get_graph().draw_mermaid())


## 9. Запуск сценария

Заявка сотрудника → весь конвейер агентов.

In [ ]:
initial_state: AgentState = {
    "request": "Командировка Москва -> Берлин, 12-15 июня, 3 ночи",
    "trace": [],
}

print(f"ЗАЯВКА: {initial_state['request']}\n" + "-" * 60)
final_state = graph.invoke(initial_state)


## 10. Обсервабилити: таймлайн шагов

Сводка по всем действиям агентов — для отладки и аудита (в проде это LangSmith/OpenTelemetry).

In [ ]:
print("ПОЛИТИКА (RAG-контекст):")
print(final_state["policy_context"])

print("\nТАЙМЛАЙН ОБСЕРВАБИЛИТИ:")
for ev in final_state["trace"]:
    print(f"  [{ev['ts']}] {ev['agent']:<18} | {ev['msg']}")

print(f"\nИтоговый выбор человека: вариант {final_state.get('human_choice')}")
print(f"Статус отправки письма : {final_state.get('email_status')}")


## 11. Итоги

- ✅ **Мультиагентная система** (Supervisor + Plan-and-Execute, исполнители по ReAct).
- ✅ **RAG внутри агента политики**: чанкинг → эмбеддинг → vector store → retrieve → rerank.
- ✅ **Поисковик предлагает варианты**, решение принимает **человек** (Human-in-the-loop).
- ✅ **Email — заглушка** (ничего реально не отправляется).
- ✅ **Планирование + действие + обсервабилити** реализованы и видны в таймлайне.

Что усилить в проде: настоящий Vector DB (FAISS/Chroma/pgvector), cross-encoder reranker,
гибрид с Knowledge Graph для лимитов по грейдам, LangSmith для трассировки,
`interrupt()` LangGraph для настоящего ожидания решения человека.